<a href="https://colab.research.google.com/github/Dhanushkotichukka/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [ ]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [ ]:
from google import genai
from pydantic import ValidationError
from pydantic_core import PydanticCustomError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    # Check for empty input and raise a ValidationError if found
    if not raw_text.strip():
        raise ValidationError(
            [
                {
                    'type': 'missing',
                    'loc': ('name',),
                    'msg': 'Field required for resume extraction, but input text is empty.',
                    'input': raw_text,
                }
            ],
            model=Resume
        )

    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash-lite',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash-lite', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [ ]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 6.8 MB/s eta 0:00:00


In [ ]:
# Load sample résumés from the lab kit
with open('/content/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 1 sample résumés

Résumé 1: Dhanushkoti Chukka — 25 skills, 0.08333333333333333 years exp

=== Full first result ===
{
  "name": "Dhanushkoti Chukka",
  "email": "23p31a0513@acet.ac.in",
  "phone": "+91 9121721128",
  "education": [
    {
      "degree": "B.Tech (C.S.E.)",
      "institution": "Aditya College of Engineering and Technology",
      "year": 2023
    },
    {
      "degree": "Class XII",
      "institution": "Andhra Pradesh State Board",
      "year": 2021
    }
  ],
  "skills": [
    "C",
    "Python",
    "Java",
    "Dart",
    "HTML",
    "CSS",
    "JavaScript",
    "Flutter",
    "Android Studio",
    "Node.js",
    "Express.js",
    "REST APIs",
    "AWS (Basics)",
    "Docker",
    "MySQL",
    "MongoDB",
    "Git",
    "GitHub",
    "VS Code",
    "Figma",
    "Data Structures",
    "OOP",
    "DBMS",
    "Operating Systems",
    "Competitive Programming"
  ],
  "projects": [
    "QLUE – AI-Powered Resume & Interview App",
    "Event Booking App"
  ],
  "ex

In [ ]:
# Empty string — should fail gracefully, not crash
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Caught gracefully: TypeError
Message: ValidationError.__new__() got an unexpected keyword argument 'model'
